# 1D Serial vs MPI on Synthetic Data

This notebook runs the synthetic-data helper script `ex_1d_serial_vs_mpi_synthetic_data.py` with real MPI, then reloads the saved rank-0 outputs into ordinary Python dictionaries so the serial and MPI results can be inspected side by side.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
OUT = ROOT / "examples" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Real MPI must be launched outside the notebook kernel.
# This helper script runs the synthetic-data case under mpirun and writes
# rank-0 outputs to disk so the notebook can load them back cleanly.
cmd = [
    "mpirun",
    "-launcher",
    "fork",
    "-n",
    "4",
    sys.executable,
    str(ROOT / "examples/python_scripts/ex_1d_serial_vs_mpi_synthetic_data.py"),
    "--output-dir",
    str(OUT),
]
completed = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True, check=True)

summary_lines = [
    line
    for line in completed.stdout.splitlines()
    if line.startswith("MPI ranks participating:")
    or line.startswith("Saved outputs to")
    or line.startswith("Largest diff =")
]
print("\n".join(summary_lines))

In [ ]:
npz_data = np.load(OUT / "ex_1d_serial_vs_mpi_synthetic_data.npz")

# Rebuild plain dictionaries so the serial and MPI results look like the
# familiar generate_structure_functions_1d outputs inside the notebook.
serial_sf = {
    "x-diffs": npz_data["x_diffs"],
    "SF_LL": npz_data["serial_SF_LL"],
    "SF_LLL": npz_data["serial_SF_LLL"],
    "SF_LTT": npz_data["serial_SF_LTT"],
    "SF_LSS": npz_data["serial_SF_LSS"],
}
mpi_sf = {
    "x-diffs": npz_data["x_diffs"],
    "SF_LL": npz_data["mpi_SF_LL"],
    "SF_LLL": npz_data["mpi_SF_LLL"],
    "SF_LTT": npz_data["mpi_SF_LTT"],
    "SF_LSS": npz_data["mpi_SF_LSS"],
}

comparison_keys = ["SF_LL", "SF_LLL", "SF_LTT", "SF_LSS"]
print(f"{'Structure function':<20} {'Max abs diff':>15}")
print("-" * 37)
for key in comparison_keys:
    diff = np.max(np.abs(serial_sf[key] - mpi_sf[key]))
    print(f"{key:<20} {diff:>15.3e}")

fig, ax = plt.subplots(figsize=(8, 5))
for name, color in zip(comparison_keys, ["C0", "C1", "C2", "C3"]):
    ax.plot(serial_sf["x-diffs"], serial_sf[name], color=color, linestyle="-", label=f"{name} serial")
    ax.plot(mpi_sf["x-diffs"], mpi_sf[name], color=color, linestyle="--", label=f"{name} mpi")
ax.set_xlabel("Separation distance")
ax.set_ylabel("Structure function")
ax.legend(ncol=2, fontsize=8)
ax.set_title("1D Serial vs MPI")
fig.tight_layout()
fig